In [61]:
from google.colab import drive
drive.mount('/content/drive')

import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [62]:
BASE_DIR = "/content/drive/MyDrive/EMBED"
MODEL_DIR = os.path.join(BASE_DIR, "models")
SPLIT_DIR = os.path.join(BASE_DIR, "splits")

MODEL_PATH = os.path.join(MODEL_DIR, "best_resnet50_embed_5yr_risk.pth")
TEST_CSV = os.path.join(SPLIT_DIR, "test_512.csv")

In [63]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet50(weights=None)

num_features = model.fc.in_features
model.fc = nn.Sequential(
    nn.Dropout(0.5),
    nn.Linear(num_features, 5)
)

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model = model.to(device)
model.eval()

print("Model loaded successfully.")

Model loaded successfully.


In [64]:
risk_cols = ["risk_1yr", "risk_2yr", "risk_3yr", "risk_4yr", "risk_5yr"]

inference_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((512, 512)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [65]:
def predict_single_image(image_path, model, transform, device):
    image = Image.open(image_path).convert("L")
    image = transform(image)
    image = image.unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(image)
        probs = torch.sigmoid(logits).cpu().numpy()[0]

    return probs

In [66]:
def predict_patient_risk(patient_images_df,uploaded_dates,model,transform,device
):
    image_results = []

    for i, (_, row) in enumerate(patient_images_df.iterrows()):

        image_path = row["processed_image_path"]

        probs = predict_single_image(
            image_path,
            model,
            transform,
            device
        )

        result = {
            "image_path": image_path,

            # USER ENTERED DATE
            "exam_date": uploaded_dates[i],

            "view": row["ViewPosition"],
            "laterality": row["ImageLateralityFinal"]
        }

        for j, col in enumerate(risk_cols):
            result[col] = probs[j] * 100

        image_results.append(result)

    image_results_df = pd.DataFrame(image_results)

    # Better aggregation
    final_risk = image_results_df[risk_cols].mean()

    return image_results_df, final_risk

In [68]:
test_df = pd.read_csv(TEST_CSV)

sample_patient = test_df["empi_anon"].iloc[0]

patient_images_df = test_df[test_df["empi_anon"] == sample_patient].head(4).copy()

print("Sample patient:", sample_patient)
print("Number of images:", len(patient_images_df))

uploaded_dates = [
    "2021-05-12",
    "2021-05-12",
    "2022-06-20",
    "2023-01-15"
]

image_results_df, final_risk = predict_patient_risk(
    patient_images_df,
    uploaded_dates,
    model,
    inference_transform,
    device
)

image_results_df

Sample patient: 88527477
Number of images: 4


,image_path,exam_date,view,laterality,risk_1yr,risk_2yr,risk_3yr,risk_4yr,risk_5yr
0,/content/drive/MyDrive/EMBED/processed_images_...,2021-05-12,MLO,R,15.997894,34.522121,48.166653,54.974110,61.124325
1,/content/drive/MyDrive/EMBED/processed_images_...,2021-05-12,MLO,L,18.413731,35.298183,48.435398,54.893547,62.523357
2,/content/drive/MyDrive/EMBED/processed_images_...,2022-06-20,MLO,R,20.416473,37.309566,48.393490,54.775818,58.927429
3,/content/drive/MyDrive/EMBED/processed_images_...,2023-01-15,CC,R,21.024324,38.014881,48.763512,53.814518,58.888634


In [69]:
final_risk_df = pd.DataFrame({
    "Risk Window": risk_cols,
    "Final Patient Risk Score (%)": final_risk.values
})

final_risk_df

,Risk Window,Final Patient Risk Score (%)
0,risk_1yr,18.963106
1,risk_2yr,36.286186
2,risk_3yr,48.439766
3,risk_4yr,54.614498
4,risk_5yr,60.365936


In [70]:
def calculate_image_contributions(image_results_df, risk_cols):
    contribution_df = image_results_df.copy()

    for col in risk_cols:
        total_risk = contribution_df[col].sum()

        if total_risk > 0:
            contribution_df[col + "_final_contribution_pct"] = (
                contribution_df[col] / total_risk
            ) * 100
        else:
            contribution_df[col + "_final_contribution_pct"] = 0

    return contribution_df

In [71]:
contribution_df = calculate_image_contributions(
    image_results_df,
    risk_cols
)

contribution_cols = [
    "exam_date",
    "view",
    "laterality",
    "risk_1yr_final_contribution_pct",
    "risk_2yr_final_contribution_pct",
    "risk_3yr_final_contribution_pct",
    "risk_4yr_final_contribution_pct",
    "risk_5yr_final_contribution_pct"
]

contribution_df[contribution_cols]

,exam_date,view,laterality,risk_1yr_final_contribution_pct,risk_2yr_final_contribution_pct,risk_3yr_final_contribution_pct,risk_4yr_final_contribution_pct,risk_5yr_final_contribution_pct
0,2021-05-12,MLO,R,21.090815,23.784616,24.859045,25.164614,25.314081
1,2021-05-12,MLO,L,24.275730,24.319298,24.997746,25.127735,25.893476
2,2022-06-20,MLO,R,26.916044,25.705074,24.976116,25.073845,24.404257
3,2023-01-15,CC,R,27.717405,26.191013,25.167088,24.633806,24.388187


In [72]:
display_cols = [
    "exam_date",
    "view",
    "laterality",
    "risk_1yr",
    "risk_2yr",
    "risk_3yr",
    "risk_4yr",
    "risk_5yr",
    "risk_5yr_final_contribution_pct"
]

contribution_df[display_cols]

,exam_date,view,laterality,risk_1yr,risk_2yr,risk_3yr,risk_4yr,risk_5yr,risk_5yr_final_contribution_pct
0,2021-05-12,MLO,R,15.997894,34.522121,48.166653,54.974110,61.124325,25.314081
1,2021-05-12,MLO,L,18.413731,35.298183,48.435398,54.893547,62.523357,25.893476
2,2022-06-20,MLO,R,20.416473,37.309566,48.393490,54.775818,58.927429,24.404257
3,2023-01-15,CC,R,21.024324,38.014881,48.763512,53.814518,58.888634,24.388187


In [73]:
test_df = pd.read_csv(TEST_CSV)

negative_patients = (
    test_df[test_df["future_risk_label"] == 0]
    ["empi_anon"]
    .unique()
)

print("Number of negative patients:", len(negative_patients))

negative_patients[:10]

Number of negative patients: 3


array([45905393, 59239977, 43414841])

In [74]:
sample_negative_patient = negative_patients[0]

print("Selected negative patient:", sample_negative_patient)

Selected negative patient: 45905393


In [78]:
negative_patient_df = test_df[test_df["empi_anon"] == sample_negative_patient].head(4).copy()

print("Number of images:",
      len(negative_patient_df))

negative_patient_df.head()

Number of images: 4


,empi_anon,acc_anon,study_date_anon,ViewPosition,ImageLateralityFinal,processed_image_path,future_risk_label,days_to_cancer,risk_1yr,risk_2yr,risk_3yr,risk_4yr,risk_5yr
23,45905393,1239318051704064,2014-11-23 00:00:00,CC,R,/content/drive/MyDrive/EMBED/processed_images_...,0,NaN,0,0,0,0,0
24,45905393,1239318051704064,2014-11-23 00:00:00,MLO,R,/content/drive/MyDrive/EMBED/processed_images_...,0,NaN,0,0,0,0,0
25,45905393,1239318051704064,2014-11-23 00:00:00,MLO,L,/content/drive/MyDrive/EMBED/processed_images_...,0,NaN,0,0,0,0,0
26,45905393,6807797065167006,2016-02-23 00:00:00,CC,R,/content/drive/MyDrive/EMBED/processed_images_...,0,NaN,0,0,0,0,0


In [79]:
uploaded_dates = [
    "2021-05-12",
    "2021-05-12",
    "2022-06-20",
    "2023-01-15"
]
negative_image_results_df, negative_final_risk = predict_patient_risk(
    negative_patient_df,
    uploaded_dates,
    model,
    inference_transform,
    device
)

In [80]:
negative_image_results_df[
    [
        "exam_date",
        "view",
        "laterality",
        "risk_1yr",
        "risk_2yr",
        "risk_3yr",
        "risk_4yr",
        "risk_5yr"
    ]
]

,exam_date,view,laterality,risk_1yr,risk_2yr,risk_3yr,risk_4yr,risk_5yr
0,2021-05-12,CC,R,19.622555,37.245159,49.671085,54.371143,59.819336
1,2021-05-12,MLO,R,16.806456,36.309261,50.273758,54.355846,62.760036
2,2022-06-20,MLO,L,18.348650,36.201904,48.301952,53.918968,62.140720
3,2023-01-15,CC,R,23.066391,38.724316,48.896553,54.872150,58.293365


In [81]:
negative_final_risk_df = pd.DataFrame({
    "Risk Window": risk_cols,
    "Final Patient Risk Score (%)": negative_final_risk.values
})

negative_final_risk_df

,Risk Window,Final Patient Risk Score (%)
0,risk_1yr,19.461012
1,risk_2yr,37.120159
2,risk_3yr,49.285835
3,risk_4yr,54.379528
4,risk_5yr,60.753365


In [82]:
for col in risk_cols:
    contribution_df[col] = contribution_df[col].round(2)
    contribution_df[col + "_final_contribution_pct"] = (
        contribution_df[col + "_final_contribution_pct"].round(2)
    )

In [84]:
import json

def frontend_ready_prediction(patient_images_df,uploaded_dates):
    image_results_df, final_risk = predict_patient_risk(
        patient_images_df,
        uploaded_dates,
        model,
        inference_transform,
        device
    )

    contribution_df = calculate_image_contributions(
        image_results_df,
        risk_cols
    )

    response = {
        "status": "success",
        "final_patient_risk": {
            "risk_1yr": round(float(final_risk["risk_1yr"]), 2),
            "risk_2yr": round(float(final_risk["risk_2yr"]), 2),
            "risk_3yr": round(float(final_risk["risk_3yr"]), 2),
            "risk_4yr": round(float(final_risk["risk_4yr"]), 2),
            "risk_5yr": round(float(final_risk["risk_5yr"]), 2),
        },
        "image_level_results": contribution_df.to_dict(orient="records")
    }

    return response

In [85]:
sample_patient = test_df["empi_anon"].iloc[10]

patient_images_df = test_df[test_df["empi_anon"] == sample_patient].head(4).copy()

uploaded_dates = [
    "2021-05-12",
    "2021-05-12",
    "2022-06-20",
    "2023-01-15"
]
response = frontend_ready_prediction(patient_images_df,uploaded_dates)

print(json.dumps(response, indent=4))

{
    "status": "success",
    "final_patient_risk": {
        "risk_1yr": 18.96,
        "risk_2yr": 36.29,
        "risk_3yr": 48.44,
        "risk_4yr": 54.61,
        "risk_5yr": 60.37
    },
    "image_level_results": [
        {
            "image_path": "/content/drive/MyDrive/EMBED/processed_images_512/88527477_4457651272914496_R_MLO_160.png",
            "exam_date": "2021-05-12",
            "view": "MLO",
            "laterality": "R",
            "risk_1yr": 15.997894287109375,
            "risk_2yr": 34.52212142944336,
            "risk_3yr": 48.16665267944336,
            "risk_4yr": 54.9741096496582,
            "risk_5yr": 61.124324798583984,
            "risk_1yr_final_contribution_pct": 21.0908145904541,
            "risk_2yr_final_contribution_pct": 23.784616470336914,
            "risk_3yr_final_contribution_pct": 24.859045028686523,
            "risk_4yr_final_contribution_pct": 25.164613723754883,
            "risk_5yr_final_contribution_pct": 25.3140811920166
   